## PACOTES ##


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import json

from pathlib import Path
from scipy.stats import chi2
from IPython.display import display

## CÓDIGO ##


In [2]:
# Caminhos do projeto
BASE_PATH = Path("thoracic_surgery_base_modelo_statsmodels.csv")
OUTPUT_DIR = Path("outputs")

# Arquivos gerados anteriormente
CAMINHO_MODELO_FINAL = OUTPUT_DIR / "modelo_final.pkl"
CAMINHO_VARIAVEIS_FINAIS = OUTPUT_DIR / "variaveis_finais.json"

# Arquivos que serão gerados neste Programa 4
CAMINHO_DIAGNOSTICO_GLOBAL = OUTPUT_DIR / "diagnostico_global.csv"
CAMINHO_HOSMER_LEMESHOW = OUTPUT_DIR / "hosmer_lemeshow.csv"

# Variável resposta
VAR_RESPOSTA = "obito_1_ano"

# Número de grupos para o teste de Hosmer-Lemeshow
G = 10

In [3]:
# Carregar base
dados = pd.read_csv(BASE_PATH)

# Carregar informações do modelo final
with open(CAMINHO_VARIAVEIS_FINAIS, "r", encoding="utf-8") as arquivo:
    info_finais = json.load(arquivo)

# Carregar modelo final ajustado
resultado_final = sm.load(str(CAMINHO_MODELO_FINAL))

print("Base, modelo final e variáveis finais carregados com sucesso.")

print("\nDimensão da base:")
print(dados.shape)

print("\nVariável resposta:")
print(VAR_RESPOSTA)

print("\nVariáveis do modelo final:")
display(pd.DataFrame({"variavel": info_finais["variaveis_modelo_final"]}))

Base, modelo final e variáveis finais carregados com sucesso.

Dimensão da base:
(470, 26)

Variável resposta:
obito_1_ano

Variáveis do modelo final:


,variavel
0,constante
1,"C(diagnostico, Treatment(reference='DGN3'))[T...."
2,"C(diagnostico, Treatment(reference='DGN3'))[T...."
3,"C(diagnostico, Treatment(reference='DGN3'))[T...."
4,"C(diagnostico, Treatment(reference='DGN3'))[T...."
5,"C(diagnostico, Treatment(reference='DGN3'))[T...."
6,"C(diagnostico, Treatment(reference='DGN3'))[T...."
7,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."
8,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."
9,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."


In [4]:
# Reconstruir variável resposta
y = dados[VAR_RESPOSTA].astype(float)

# Reconstruir matriz de covariáveis do modelo final
variaveis_modelo_final = info_finais["variaveis_modelo_final"]

X_final = dados[variaveis_modelo_final].copy()

# Garantir formato numérico
for coluna in X_final.columns:
    X_final[coluna] = pd.to_numeric(X_final[coluna], errors="coerce")

# Verificar valores ausentes
faltantes = X_final.isna().sum()
faltantes = faltantes[faltantes > 0]

if len(faltantes) > 0:
    print("Variáveis com valores ausentes:")
    display(faltantes.to_frame("n_faltantes"))
    raise ValueError("Existem valores ausentes em X_final.")

print("Dimensão de y:")
print(y.shape)

print("\nDimensão de X_final:")
print(X_final.shape)

print("\nPrimeiras linhas de X_final:")
display(X_final.head())

Dimensão de y:
(470,)

Dimensão de X_final:
(470, 13)

Primeiras linhas de X_final:


,constante,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]","C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12]","C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC13]","C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC14]",dispneia_antes_cirurgia,diabetes_mellitus_tipo_2,tabagismo
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [5]:
# Probabilidades ajustadas pelo modelo final
probabilidades_ajustadas = resultado_final.predict(X_final)

tabela_probabilidades = pd.DataFrame({
    "obito_observado": y,
    "probabilidade_ajustada": probabilidades_ajustadas
})

print("Resumo das probabilidades ajustadas:")
display(tabela_probabilidades["probabilidade_ajustada"].describe().to_frame())

print("\nPrimeiras linhas:")
display(tabela_probabilidades.head())

Resumo das probabilidades ajustadas:


,probabilidade_ajustada
count,4.700000e+02
mean,1.489362e-01
std,1.176543e-01
min,1.980120e-10
25%,8.469694e-02
50%,1.193406e-01
75%,1.577872e-01
max,7.479592e-01



Primeiras linhas:


,obito_observado,probabilidade_ajustada
0,0.0,0.465570
1,0.0,0.119341
2,0.0,0.084697
3,0.0,0.026848
4,1.0,0.084697


In [7]:
# Diagnóstico global pela deviance

deviance_final = float(resultado_final.deviance)
gl_residuo = int(resultado_final.df_resid)
p_valor_deviance = float(chi2.sf(deviance_final, gl_residuo))

diagnostico_deviance = pd.DataFrame({
    "medida": [
        "deviance_modelo_final",
        "graus_liberdade_residuo",
        "p_valor_deviance",
        "log_verossimilhanca",
        "numero_observacoes",
        "numero_parametros"
    ],
    "valor": [
        deviance_final,
        gl_residuo,
        p_valor_deviance,
        float(resultado_final.llf),
        int(resultado_final.nobs),
        int(len(resultado_final.params))
    ]
})

display(diagnostico_deviance)

,medida,valor
0,deviance_modelo_final,351.894284
1,graus_liberdade_residuo,457.000000
2,p_valor_deviance,0.999916
3,log_verossimilhanca,-175.947142
4,numero_observacoes,470.000000
5,numero_parametros,13.000000


In [8]:
# Teste de Hosmer-Lemeshow

dados_hl = pd.DataFrame({
    "y": y,
    "prob": probabilidades_ajustadas
})

# Criar grupos com base nas probabilidades ajustadas
dados_hl["grupo"] = pd.qcut(
    dados_hl["prob"],
    q=G,
    duplicates="drop"
)

# Tabela observados e esperados por grupo
tabela_hl = (
    dados_hl
    .groupby("grupo", observed=False)
    .agg(
        n=("y", "count"),
        obitos_observados=("y", "sum"),
        obitos_esperados=("prob", "sum"),
        probabilidade_media=("prob", "mean")
    )
    .reset_index()
)

# Não óbitos observados e esperados
tabela_hl["nao_obitos_observados"] = tabela_hl["n"] - tabela_hl["obitos_observados"]
tabela_hl["nao_obitos_esperados"] = tabela_hl["n"] - tabela_hl["obitos_esperados"]

# Componentes da estatística de Hosmer-Lemeshow
tabela_hl["componente_obito"] = (
    (tabela_hl["obitos_observados"] - tabela_hl["obitos_esperados"]) ** 2
    / tabela_hl["obitos_esperados"]
)

tabela_hl["componente_nao_obito"] = (
    (tabela_hl["nao_obitos_observados"] - tabela_hl["nao_obitos_esperados"]) ** 2
    / tabela_hl["nao_obitos_esperados"]
)

# Tratar possíveis divisões por zero
tabela_hl = tabela_hl.replace([np.inf, -np.inf], np.nan)
tabela_hl[["componente_obito", "componente_nao_obito"]] = (
    tabela_hl[["componente_obito", "componente_nao_obito"]].fillna(0)
)

# Estatística final
estatistica_hl = float(
    tabela_hl["componente_obito"].sum()
    + tabela_hl["componente_nao_obito"].sum()
)

g_efetivo = tabela_hl.shape[0]
gl_hl = g_efetivo - 2
p_valor_hl = float(chi2.sf(estatistica_hl, gl_hl))

print("Tabela do teste de Hosmer-Lemeshow:")
display(tabela_hl)

print("\nResultado do teste de Hosmer-Lemeshow:")
print(f"Estatística HL: {estatistica_hl:.6f}")
print(f"Graus de liberdade: {gl_hl}")
print(f"p-valor: {p_valor_hl:.6f}")

Tabela do teste de Hosmer-Lemeshow:


,grupo,n,obitos_observados,obitos_esperados,probabilidade_media,nao_obitos_observados,nao_obitos_esperados,componente_obito,componente_nao_obito
0,"(-0.000999999802, 0.0388]",54,1.0,1.654787,0.030644,53.0,52.345213,0.259094,0.008191
1,"(0.0388, 0.0847]",123,8.0,10.018358,0.081450,115.0,112.981642,0.406630,0.036057
2,"(0.0847, 0.113]",14,1.0,1.555409,0.111101,13.0,12.444591,0.198326,0.024788
3,"(0.113, 0.119]",133,16.0,15.872304,0.119341,117.0,117.127696,0.001027,0.000139
4,"(0.119, 0.13]",8,2.0,1.043596,0.130450,6.0,6.956404,0.876497,0.131492
5,"(0.13, 0.18]",46,10.0,7.752480,0.168532,36.0,38.247520,0.651578,0.132070
6,"(0.18, 0.298]",49,15.0,12.800071,0.261226,34.0,36.199929,0.378099,0.133693
7,"(0.298, 0.748]",43,17.0,19.302997,0.448907,26.0,23.697003,0.274765,0.223817



Resultado do teste de Hosmer-Lemeshow:
Estatística HL: 3.736264
Graus de liberdade: 6
p-valor: 0.712316


In [9]:
# Resumo do diagnóstico global

diagnostico_global = pd.DataFrame({
    "medida": [
        "deviance_modelo_final",
        "graus_liberdade_deviance",
        "p_valor_deviance",
        "estatistica_hosmer_lemeshow",
        "graus_liberdade_hosmer_lemeshow",
        "p_valor_hosmer_lemeshow",
        "numero_grupos_hosmer_lemeshow",
        "log_verossimilhanca",
        "numero_observacoes",
        "numero_parametros"
    ],
    "valor": [
        deviance_final,
        gl_residuo,
        p_valor_deviance,
        estatistica_hl,
        gl_hl,
        p_valor_hl,
        g_efetivo,
        float(resultado_final.llf),
        int(resultado_final.nobs),
        int(len(resultado_final.params))
    ]
})

display(diagnostico_global)

,medida,valor
0,deviance_modelo_final,351.894284
1,graus_liberdade_deviance,457.000000
2,p_valor_deviance,0.999916
3,estatistica_hosmer_lemeshow,3.736264
4,graus_liberdade_hosmer_lemeshow,6.000000
5,p_valor_hosmer_lemeshow,0.712316
6,numero_grupos_hosmer_lemeshow,8.000000
7,log_verossimilhanca,-175.947142
8,numero_observacoes,470.000000
9,numero_parametros,13.000000


In [10]:
# Salvar resultados

diagnostico_global.to_csv(
    CAMINHO_DIAGNOSTICO_GLOBAL,
    index=False,
    encoding="utf-8-sig"
)

tabela_hl.to_csv(
    CAMINHO_HOSMER_LEMESHOW,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos com sucesso:")
print(f"- {CAMINHO_DIAGNOSTICO_GLOBAL}")
print(f"- {CAMINHO_HOSMER_LEMESHOW}")

Arquivos salvos com sucesso:
- outputs\diagnostico_global.csv
- outputs\hosmer_lemeshow.csv
